# Edge basics

In [ ]:
# barebones version of edge pipeline 
# works out sobel and canny for every image turns them into simple features
# and prints what the features look like for each class
# no svm yet

# how to run
#   keep this in the edge folder next to the Dataset folder and folds csv
#   python edge_basics.py

# order of the file
#   1 settings
#   2 load data the same way as the setup notebook
#   3 preprocessing
#   4 sobel and canny
#   5 features
#   6 compute features for every image
#   7 print results
#   8 example picture

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

## 1 settings

In [ ]:

# 1 settings

# folder with the Dataset folder and folds csv in it
DATA_ROOT = Path.cwd().parent
FOLDS_CSV = DATA_ROOT / "folds.csv"

# class names in id order same as the setup notebook
CLASS_NAMES = ["Crack", "Pothole", "Surface Erosion"]

# original image size and size of image we want to use (256)
IMAGE_SIZE = 640
EDGE_SIZE = 256

# light blur before sobel for noise reduction
BLUR_SIGMA = 1.0

# canny thresholds
CANNY_LOW = 50
CANNY_HIGH = 150

# 9 bins over 0 to 180 degrees so each bin is 20 degrees
N_BINS = 9

# 2 means split the image into 4 boxes
GRID = 2

# set to a number like 20 for a quick run or None for every image
QUICK_TEST_PER_CLASS = None

# where outputs get saved
OUTPUT_DIR = Path.cwd() / "basics_output"

## 2 load data the same way as the setup notebook

In [ ]:
# 2 load data the same way as the setup notebook

def load_table():
    # folds csv has columns path class_name class_id fold
    table = pd.read_csv(FOLDS_CSV)
    table["full_path"] = [str(DATA_ROOT / p) for p in table["path"]]

    # optional smaller run
    if QUICK_TEST_PER_CLASS is not None:
        pieces = []
        for class_id in range(len(CLASS_NAMES)):
            rows = table[table["class_id"] == class_id]
            pieces.append(rows.sample(min(len(rows), QUICK_TEST_PER_CLASS), random_state=42))
        table = pd.concat(pieces)

    return table.reset_index(drop=True)


def load_image(path):
    # same as load image in the setup notebook
    # rgb and if not 640 by 640 cut the middle square out then resize
    with Image.open(path) as im:
        img = im.convert("RGB")
    w, h = img.size
    if (w, h) != (IMAGE_SIZE, IMAGE_SIZE):
        side = min(w, h)
        left = (w - side) // 2
        top = (h - side) // 2
        img = img.crop((left, top, left + side, top + side)).resize((IMAGE_SIZE, IMAGE_SIZE))
    return np.array(img)

## 3 preprocessing

In [ ]:
# 3 preprocessing

def prepare_image(rgb):
    # gray then smaller then a light blur
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, (EDGE_SIZE, EDGE_SIZE), interpolation=cv2.INTER_AREA)
    if BLUR_SIGMA > 0:
        gray = cv2.GaussianBlur(gray, (0, 0), BLUR_SIGMA)
    return gray

## 4 sobel and canny

In [ ]:
# 4 sobel and canny

def sobel_gradients(gray):
    # change in brightness left to right and top to bottom
    gray = gray.astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    # how strong the edge is and which way it points
    # angle folded into 0 to 180 because dark to light and light to dark is the same line
    magnitude = np.sqrt(gx ** 2 + gy ** 2)
    angle = np.degrees(np.arctan2(gy, gx)) % 180
    return magnitude, angle


def canny_edges(gray):
    # True where canny thinks there is an edge
    return cv2.Canny(gray, CANNY_LOW, CANNY_HIGH) > 0

## 5 features

In [ ]:
# 5 features

def split_into_grid(image):
    # cut into GRID by GRID boxes left to right then top to bottom
    h, w = image.shape[:2]
    boxes = []
    for row in range(GRID):
        for col in range(GRID):
            boxes.append(image[row * h // GRID:(row + 1) * h // GRID, col * w // GRID:(col + 1) * w // GRID])
    return boxes


def orientation_histogram(magnitude, angle):
    # each pixel votes for the bin its angle is in and stronger edges vote more
    hist, _ = np.histogram(angle, bins=N_BINS, range=(0, 180), weights=magnitude)

    # make it add up to 1 so overall brightness doesnt matter
    total = hist.sum()
    if total > 0:
        hist = hist / total
    return hist


def image_features(rgb):
    gray = prepare_image(rgb)
    magnitude, angle = sobel_gradients(gray)
    edges = canny_edges(gray)

    # sobel only features
    # average edge strength is one number for the whole image
    edge_strength = float(magnitude.mean())
    # one histogram for the whole image is 9 numbers
    whole_hist = orientation_histogram(magnitude, angle)
    # one histogram per box is 4 times 9 so 36 numbers
    grid_hist = np.concatenate([
        orientation_histogram(m, a) for m, a in zip(split_into_grid(magnitude), split_into_grid(angle))
    ])

    # canny only features
    # fraction of pixels that are edges in the whole image is one number
    edge_density = edges.mean()
    # fraction per box is 4 numbers
    grid_density = np.array([box.mean() for box in split_into_grid(edges)])

    return {
        "edge_strength": edge_strength,
        "edge_density": edge_density,
        "whole_hist": whole_hist,
        "grid_hist": grid_hist,
        "grid_density": grid_density,
    }

## 6 compute features for every image

In [ ]:
# 6 compute features for every image

def compute_all(table):
    results = []
    for i, path in enumerate(table["full_path"]):
        results.append(image_features(load_image(path)))
        if (i + 1) % 200 == 0:
            print(f"  done {i + 1} of {len(table)}")
    print(f"  done all {len(table)} images")
    print()
    return results

## 7 print results

In [ ]:
# 7 print results

def bin_labels():
    # names like 0-20 20-40 for each bin
    width = 180 // N_BINS
    return [f"{b * width}-{(b + 1) * width}" for b in range(N_BINS)]


def print_results(table, results):
    # what we loaded
    print("images per class")
    print(table["class_name"].value_counts().reindex(CLASS_NAMES).to_string())
    print()

    # how many numbers each feature gives per image
    print("numbers per image for each feature")
    for name, value in results[0].items():
        print(f"  {name}: {np.size(value)}")
    print()

    # one row per image for the single number features
    simple = pd.DataFrame({
        "class_name": table["class_name"],
        "edge_strength": [r["edge_strength"] for r in results],
        "edge_density": [r["edge_density"] for r in results],
    })

    # average and spread of each single number feature per class
    print("sobel edge strength and canny edge density per class mean and std")
    print(simple.groupby("class_name").agg(["mean", "std"]).reindex(CLASS_NAMES).round(3).to_string())
    print()

    # average orientation histogram per class
    # shows which edge directions are most common in each class
    hists = np.array([r["whole_hist"] for r in results])
    average_hist = pd.DataFrame(
        [hists[table["class_name"] == name].mean(axis=0) for name in CLASS_NAMES],
        index=CLASS_NAMES,
        columns=bin_labels(),
    )
    print("average sobel orientation histogram per class in degrees")
    print(average_hist.round(3).to_string())
    print()

    # save everything as one row per image so we can look at it later or feed an svm
    OUTPUT_DIR.mkdir(exist_ok=True)
    rows = []
    for (_, row), r in zip(table.iterrows(), results):
        out = {"path": row["path"], "class_name": row["class_name"], "class_id": row["class_id"], "fold": row["fold"],
               "edge_strength": r["edge_strength"], "edge_density": r["edge_density"]}
        out.update({f"whole_hist_{i}": v for i, v in enumerate(r["whole_hist"])})
        out.update({f"grid_hist_{i}": v for i, v in enumerate(r["grid_hist"])})
        out.update({f"grid_density_{i}": v for i, v in enumerate(r["grid_density"])})
        rows.append(out)
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "features.csv", index=False)
    print("saved features to", OUTPUT_DIR / "features.csv")

## 8 example picture

In [ ]:

# 8 example picture
# one image per class showing the original sobel strength and canny edges

def save_example_picture(table):
    fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(9, 3 * len(CLASS_NAMES)))

    for row, name in enumerate(CLASS_NAMES):
        path = table[table["class_name"] == name]["full_path"].iloc[0]
        rgb = load_image(path)
        gray = prepare_image(rgb)
        magnitude, _ = sobel_gradients(gray)
        edges = canny_edges(gray)

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"{name}")
        axes[row, 1].imshow(magnitude, cmap="gray")
        axes[row, 1].set_title("sobel magnitude")
        axes[row, 2].imshow(edges, cmap="gray")
        axes[row, 2].set_title("canny edges")
        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    OUTPUT_DIR.mkdir(exist_ok=True)
    plt.savefig(OUTPUT_DIR / "examples.png", dpi=120)
    print("saved picture to", OUTPUT_DIR / "examples.png")

## run

In [ ]:
print("loading data from", FOLDS_CSV)
table = load_table()

In [ ]:
print("working out sobel and canny features")
results = compute_all(table)

In [ ]:
print_results(table, results)

In [ ]:
save_example_picture(table)